# Licensing recommendation

Run **`uv run streamvault`** from the project root to fetch missing titles and get the recommendation, rankings workbook, and exclusions report in `out/`. This notebook uses the same analysis on the saved files. It writes its recommendation, workbook, exclusions report, and run record together in `data/recommendation-report/`.

The two windows cover all dated, released films and films released from 1 January 2000 onward. Each installment uses its own date. The [README](../README.md) has the run instructions and export list, and the [decision notes](../docs/DECISIONS.md) explain the scoring rules.

## 1. Recommendation and shortlists

The recommendation uses all eligible bundles and standalones. A bundle's score is the middle score of its observed films. Every in-window installment needs a score for its bundle to qualify. A vote-adjusted score gives films with fewer votes less weight against a shared 7.0 starting point.

In [1]:
import pandas as pd
from IPython.display import Markdown, display

from streamvault.config import CONFIG
from streamvault.recommendation_run import run_recommendation

# Call the same analysis and report writer as the CLI after acquisition.
analysis = run_recommendation(CONFIG)
inputs = analysis.inputs
results = analysis.results
scoring = analysis.scoring_tests
display(Markdown(analysis.summary))

# StreamVault recommendation

The candidate list has 127 rows and 112 resolved films. The exclusions report gives a reason for every source row that did not contribute a film.

- **All years** — Favors standalones
- **2000 onward** — Favors standalones

Scores are out of 10 and give less weight to ratings with fewer votes.

## All years

Compared 32 eligible bundles and 16 standalones. 112 films entered this window.
The middle bundle score was 7.59 and the middle standalone score was 8.36. The estimated 95% range for the gap (bundle minus standalone) was -1.15 to -0.49 rating points.
With each eligible bundle's strongest film set aside, the result was **Favors standalones**.

### Franchise bundles (3 of 3)

1. **The Godfather Collection** — 8.47. Median of 3 observed installments. Lowest score 7.36.
2. **The Lord of the Rings Collection** — 8.40. Median of 3 observed installments. Lowest score 8.38.
3. **The Hannibal Lecter Collection** — 8.28. One observed installment. Evidence covers that film.

### Standalones (5 of 5)

1. **The Shawshank Redemption** — 8.68. Audience rating 8.73 from 31,864 votes.
2. **Schindler&#x27;s List** — 8.49. Audience rating 8.57 from 18,194 votes.
3. **Spirited Away** — 8.46. Audience rating 8.54 from 19,231 votes.
4. **Interstellar** — 8.46. Audience rating 8.49 from 41,925 votes.
5. **Pulp Fiction** — 8.43. Audience rating 8.48 from 31,350 votes.

### Talent evidence

- Director **Christopher Nolan** — 7 films across 5 film groups. The rating gap stayed at +0.20 after removing the largest group.
- Director **Steven Spielberg** — 8 films across 4 film groups. The rating gap stayed at +0.05 after removing the largest group.
- Lead cast **Harrison Ford** — 8 films across 3 film groups. The rating gap stayed at +0.31 after removing the largest group.

## 2000 onward

Compared 24 eligible bundles and 8 standalones. 71 films entered this window.
The middle bundle score was 7.15 and the middle standalone score was 8.14. The estimated 95% range for the gap (bundle minus standalone) was -1.45 to -0.50 rating points.
With each eligible bundle's strongest film set aside, the result was **Favors standalones**.

### Franchise bundles (3 of 3)

1. **The Lord of the Rings Collection** — 8.40. Median of 3 observed installments. Lowest score 8.38.
2. **The Avengers Collection** — 8.13. Median of 4 observed installments. Lowest score 7.27.
3. **Kill Bill Collection** — 7.87. Median of 2 observed installments. Lowest score 7.82.

### Standalones (5 of 5)

1. **Spirited Away** — 8.46. Audience rating 8.54 from 19,231 votes.
2. **Interstellar** — 8.46. Audience rating 8.49 from 41,925 votes.
3. **Inception** — 8.37. Audience rating 8.40 from 40,754 votes.
4. **The Prestige** — 8.15. Audience rating 8.21 from 18,478 votes.
5. **Memento** — 8.13. Audience rating 8.20 from 17,054 votes.

### Talent evidence

- Director **Christopher Nolan** — 7 films across 5 film groups. The rating gap stayed at +0.12 after removing the largest group.
- Lead cast **Michael Caine** — 3 films across 3 film groups. The rating gap stayed at +0.53 after removing the largest group.

No lead actor stayed ahead of comparable films in both release windows.

## Evidence limits

- The comparison covers the supplied list. Unresolved films could change the results.
- Each bundle represents the installments observed in that window. The studio package may contain more films.
- Scores come from TMDB audience ratings. Historical box office gives context. Kwame needs licensing quotes to check what fits the $14M budget.
- Open the rankings workbook for both windows side by side and the exclusions report for missing or unusable evidence.


## 2. What went into the decision?

The source counts account for every candidate CSV row. The release-window counts start from the resolved films. See the [exclusions report](../data/recommendation-report/recommendation_exclusions.md) for individual reasons and data issues.

In [2]:
source_summary = inputs.source_counts.rename_axis("CSV outcome").rename("Rows").to_frame()
display(source_summary)
display(pd.DataFrame(inputs.window_counts))

# A series can have fewer observed installments when the older releases are left out.
if not inputs.changed_bundles.empty:
    display(Markdown("**Bundles with observed films on both sides of the cutoff**"))
    display(inputs.changed_bundles.rename_axis("Collection ID"))

,Rows
CSV outcome,
Cached,112
Duplicate TMDB movie ID; skipped.,6
TMDB lookup: TMDB returned HTTP 404.,5
Not a positive numeric TMDB movie ID.,3
Blank row; no TMDB movie ID.,1


,All years,2000 onward
Resolved films,112,112
Missing release date,0,0
Dated and missing release status,0,0
Dated and marked with another status,0,0
Dated and released but before the cutoff,0,41
Films in window,112,71
Collection-linked films,96,63
Standalone films,16,8
Observed collections,32,24
Collections with two or more observed films,23,17


**Bundles with observed films on both sides of the cutoff**

,Bundle,Before 2000,All years,2000 onward
Collection ID,,,,
10,Star Wars Collection,4,6,2
84,Indiana Jones Collection,3,4,1
328,Jurassic Park Collection,2,3,1
528,The Terminator Collection,2,4,2
1570,Die Hard Collection,3,4,1
2344,The Matrix Collection,1,3,2
10194,Toy Story Collection,2,4,2
87359,Mission: Impossible Collection,1,5,4


## 3. What could change the decision?

The main comparison counts one median score per eligible bundle and one score per eligible standalone. A negative gap means the middle standalone score is higher. The whole estimated range must clear the 0.25-point margin before the ratings favor either group.

The film-by-film view shows what happens when larger franchises contribute more scores. The strongest-film test checks whether the decision depends on the highest-scoring entry in each bundle.

In [3]:
# These views read saved results. Rounding changes the display only.
decision_fields = {
    "bundle_units": "Eligible bundles", "standalone_units": "Eligible standalones",
    "bundle_median": "Middle bundle score", "standalone_median": "Middle standalone score",
    "rating_gap": "Bundle minus standalone", "interval_low": "95% range low", "interval_high": "95% range high",
    "conclusion": "Ratings conclusion",
}
display(results.decision_comparison[list(decision_fields)].rename(columns=decision_fields).round(3))

display(Markdown("**Compare all scored films one by one**"))
film_fields = {
    "bundle_units": "Scored collection films", "standalone_units": "Scored standalones",
    "rating_gap": "Collection film minus standalone",
}
display(results.film_comparison[list(film_fields)].rename(columns=film_fields).round(3))

display(Markdown("**Leave out each eligible bundle's strongest film**"))
omission_fields = {
    "bundles_lost": "Bundles that disappear", "bundles_left_with_one": "Bundles left with one film",
    "rating_gap": "Bundle minus standalone", "gap_change": "Change in gap", "conclusion": "Ratings conclusion",
}
display(results.omission_comparison[list(omission_fields)].rename(columns=omission_fields).round(3))

,Eligible bundles,Eligible standalones,Middle bundle score,Middle standalone score,Bundle minus standalone,95% range low,95% range high,Ratings conclusion
window,,,,,,,,
All years,32,16,7.586,8.360,-0.774,-1.149,-0.491,Favors standalones
2000 onward,24,8,7.150,8.142,-0.991,-1.454,-0.497,Favors standalones


**Compare all scored films one by one**

,Scored collection films,Scored standalones,Collection film minus standalone
window,,,
All years,96,16,-0.796
2000 onward,63,8,-0.738


**Leave out each eligible bundle's strongest film**

,Bundles that disappear,Bundles left with one film,Bundle minus standalone,Change in gap,Ratings conclusion
window,,,,,
All years,9,1,-1.078,-0.304,Favors standalones
2000 onward,7,5,-1.165,-0.173,Favors standalones


In [4]:
# The workbook compares baseline ranks. Scoring alternatives stay in the notebook and run record.
display(Markdown("**Change one scoring rule at a time**"))
test_fields = ["scenario", "window", "rating_gap", "conclusion", "status"]
display(scoring.comparisons[test_fields].round(3))
cutoff_changes = scoring.shortlist_changes["scenario"].eq("2000 cutoff").sum()
print(f"The date cutoff changes {cutoff_changes} shortlist positions.")

display(Markdown("**Historical gross per film**"))
display(scoring.gross_comparison.rename(columns={
    "window": "Window", "collection_films": "Collection films with gross",
    "standalone_films": "Standalones with gross", "collection_median": "Middle collection film gross ($)",
    "standalone_median": "Middle standalone gross ($)",
}).round(0))

**Change one scoring rule at a time**

,scenario,window,rating_gap,conclusion,status
0,Baseline,All years,-0.774,Favors standalones,Stable
1,Baseline,2000 onward,-0.991,Favors standalones,Stable
2,Reach first,All years,-0.774,Favors standalones,Stable
3,Reach first,2000 onward,-0.991,Favors standalones,Stable
4,At least 500 votes,All years,-0.774,Favors standalones,Stable
5,At least 500 votes,2000 onward,-0.991,Favors standalones,Stable
6,"At least 5,000 votes",All years,-0.743,Favors standalones,Stable
7,"At least 5,000 votes",2000 onward,-0.877,Favors standalones,Stable
8,6.5 rating anchor,All years,-0.782,Favors standalones,Stable
9,6.5 rating anchor,2000 onward,-0.995,Favors standalones,Stable


The date cutoff changes 13 shortlist positions.


**Historical gross per film**

,Window,Collection films with gross,Standalones with gross,Middle collection film gross ($),Middle standalone gross ($)
0,All years,96,16,578937111.0,161802536.0
1,2000 onward,63,8,786636033.0,283195048.0


## 4. Genre and talent evidence

Each series counts as one film group and each standalone as its own group. Look for ratings above the benchmark across at least three films and two groups, then remove each person's largest group and check the direction again. Comparisons describe patterns in this candidate list.

In [5]:
coverage_fields = {
    "window": "Window", "role": "Role", "films_without_credits": "Films without role credits",
    "three_films_one_group": "People with 3+ films in one series",
    "eligible_across_groups": "People with 3+ films across groups",
    "rating_above_both": "People above the benchmark in both checks",
}
display(analysis.talent.coverage[list(coverage_fields)].rename(columns=coverage_fields))

# Compare model errors on the same held-aside films with and without genre tags.
model_fields = {
    "window": "Window", "outcome": "Outcome", "benchmark_films": "Films benchmarked",
    "genre_terms": "Genre tags used", "controls_error": "Error without genres", "genre_error": "Error with genres",
}
display(analysis.talent.benchmarks[list(model_fields)].rename(columns=model_fields).round(3))

,Window,Role,Films without role credits,People with 3+ films in one series,People with 3+ films across groups,People above the benchmark in both checks
0,All years,Director,0,7,6,2
1,All years,Lead cast,0,32,12,1
2,2000 onward,Director,0,3,2,1
3,2000 onward,Lead cast,0,21,6,1


,Window,Outcome,Films benchmarked,Genre tags used,Error without genres,Error with genres
0,All years,Audience rating,112,10,0.474,0.466
1,All years,Log gross,112,10,0.217,0.207
2,2000 onward,Audience rating,71,5,0.553,0.591
3,2000 onward,Log gross,71,5,0.157,0.163


## 5. Reports and timing

Use `rankings.xlsx` to compare the two windows. It has a tab for each kind of option, and the bundle tab names the observed films. The exclusions report lists any film data issues. Keep the input CSV, cache, removal log, any acquisition record, and `run.json` from the same run to repeat it.

This notebook can display its saved 500-title runtime test. To refresh the notebook's runtime and memory reports in `data/recommendation-report/benchmark/`, run the [benchmark commands with `--notebook`](../docs/benchmarks.md#rerun-the-benchmarks).

**For a closer look**, run the relevant example in a new cell. Each uses this run's results. Change `"All years"` to `"2000 onward"` to view the newer window.

| Evidence | Example |
| --- | --- |
| Scores and financial amounts for films in shortlisted bundles | `display(results.shortlisted_bundle_films["All years"])` |
| Which film was set aside in each bundle's strongest-film test | `display(results.omission_details["All years"])` |
| Which individual ranks moved when a scoring rule or window changed | `display(scoring.shortlist_changes)` |
| Individual talent results | `display(analysis.talent.people["All years"])` |
| Genre summaries and shared films | `display(analysis.talent.genres["All years"], analysis.talent.genre_overlaps["All years"])` |

In [6]:
record = analysis.record
print(f"{len(record['checks'])} data checks passed.")
print(f"Recommendation: {analysis.directory / 'recommendation.md'}")
print(f"Full rankings: {analysis.rankings_path}")
print(f"Exclusions report: {analysis.exclusions_path}")
print(f"Fetch timing: {record['acquisition']['status']}")
display(Markdown(record["cached_timing_boundary"]))

# Add the benchmark columns when the saved measurement matches this code and snapshot.
timing = pd.DataFrame({"This run (s)": pd.Series(record["cached_stage_seconds"])})
scale = record["cached_scale_check"]
if scale is not None:
    measured = pd.DataFrame(scale["stages"]).set_index("stage")
    timing = timing.join(measured[["projected_seconds", "measured_seconds"]].rename(columns={
        "projected_seconds": "Estimate for 500 (s)", "measured_seconds": "Measured 500 (s)",
    }), validate="one_to_one")
timing.loc["Total"] = timing.sum()
display(timing.rename_axis("Part of the analysis").round(3))

32 data checks passed.
Recommendation: /home/faazeleishola/training/streamvault/data/recommendation-report/recommendation.md
Full rankings: /home/faazeleishola/training/streamvault/data/recommendation-report/rankings.xlsx
Exclusions report: /home/faazeleishola/training/streamvault/data/recommendation-report/recommendation_exclusions.md
Fetch timing: Matched acquisition run


From reading the saved analysis inputs through writing report files and checking fingerprints. TMDB fetching, test-cache setup, and writing the final run record happen outside this timing.

,This run (s),Estimate for 500 (s),Measured 500 (s)
Part of the analysis,,,
Read candidate list and cache,0.154,0.778,0.701
Check saved inputs,0.033,0.119,0.103
Normalize and check films,0.549,2.047,2.118
Reconcile source rows and set release windows,0.033,0.102,0.046
"Score, rank, and compare eligible options",0.152,0.421,0.163
Test removing the strongest films,0.074,0.235,0.079
Account for every film's shortlist outcome,0.023,0.075,0.021
Review genre and talent,0.547,1.539,0.541
Test scoring alternatives,0.537,1.478,0.456
